# Explore any design
Reads only the CSV results of the study set in `config.yaml` (or the file in `WIND_BLADE_CONFIG`), so run `01` to `03` first. Pick a `design_id` from `03_performance_summary.csv` in that results folder.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import bem
cfg = bem.load_config()
R = cfg["paths"]["results"]
RAD = cfg["rotor"]["R"]
print("results folder:", R)
perf = pd.read_csv(R / "03_performance_summary.csv")
perf.head(15)[["rank","design_id","Cp_at_lam_des_TL","Cp_max_TL","lam_Cp_max_TL","Cp_max_noTL","clamped_fraction"]]

In [ ]:
# filter, e.g. one airfoil
foil = perf.iloc[0].airfoil   # edit this
perf[perf.airfoil == foil].sort_values("rank").head(10)

In [ ]:
design_ids = list(perf.sort_values("rank").drop_duplicates("airfoil").design_id.head(2))   # edit this list

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for did in design_ids:
    d = pd.read_csv(R / "designs" / f"{did}.csv")
    b = pd.read_csv(R / "blades" / f"{did}.csv")
    c = pd.read_csv(R / "performance" / f"{did}.csv")
    l, = ax[0, 0].plot(b.r / RAD, b.chord * 1000, lw=2, label=did)
    ax[0, 0].plot(d.r_R, d.chord * 1000, ls=":", color=l.get_color())
    ax[0, 1].plot(b.r / RAD, b.theta_deg, lw=2, color=l.get_color())
    ax[1, 0].plot(c.lam, c.Cp_TL, lw=2, color=l.get_color()); ax[1, 0].plot(c.lam, c.Cp_noTL, lw=1, alpha=.5, color=l.get_color())
    ax[1, 1].plot(c.lam, c.Ct_TL, lw=2, color=l.get_color()); ax[1, 1].plot(c.lam, c.Ct_noTL, lw=1, alpha=.5, color=l.get_color())
ax[0, 0].set(xlabel="r/R [-]", ylabel="chord [mm]", title="chord (dotted: ideal BEM)")
ax[0, 1].set(xlabel="r/R [-]", ylabel="twist [deg]")
ax[1, 0].set(xlabel="λ [-]", ylabel="$C_P$ [-]", title="thick: tip loss, thin: no tip loss")
ax[1, 1].set(xlabel="λ [-]", ylabel="$C_T$ [-]")
ax[0, 0].legend(fontsize=7)
for a in ax.flat: a.grid(alpha=.3)
fig.tight_layout()

In [ ]:
# polar of one airfoil at all Re (converged XFOIL points)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for f in sorted((R / "polars" / foil).glob("raw_Re*.csv")):
    p = pd.read_csv(f); lab = f.stem.replace("raw_", "")
    ax[0].plot(p.alpha, p.CL / p.CD, label=lab); ax[1].plot(p.CL, p.CL / p.CD)
ax[0].set(xlabel="α [deg]", ylabel="$C_L/C_D$ [-]"); ax[1].set(xlabel="$C_L$ [-]", ylabel="$C_L/C_D$ [-]")
ax[0].legend(fontsize=7)